In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 14.3 MB/s eta 0:00:00


In [7]:
### Load ttl file Version 5 ###

from rdflib import Graph
from rdflib.plugins.parsers.notation3 import BadSyntax

g = Graph()
try:
    g.parse("/content/drive/MyDrive/UU/Period2/KDE/Assignement/databaseV5.ttl", format="turtle")
    print("OK. Triples:", len(g))
except BadSyntax as e:
    print("BadSyntax:", e)
except Exception as e:
    print(type(e).__name__, e)

OK. Triples: 626


In [22]:
### Number of Diseases in tll file ###
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT (COUNT(DISTINCT ?d) AS ?total_diseases)
WHERE {
  ?d a ex:Disease ;
     skos:prefLabel ?label;
}
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('24', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),)


In [24]:
### Number of Symptoms in tll file ###
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT (COUNT(DISTINCT ?s) AS ?total_symptoms)
WHERE {
  ?d a ex:Disease .
  ?d ex:hasPrimarySymptom|ex:hasSecondarySymptom ?s .
}
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('115', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),)


In [8]:
### 5 Diseases with least symptoms ###
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label ?d (COUNT(?s) AS ?matched)
WHERE {
  ?d a ex:Disease ;
     skos:prefLabel ?label;
     ex:hasPrimarySymptom|ex:hasSecondarySymptom ?s .
}
GROUP BY ?d
ORDER BY ASC(?matched)
LIMIT 5
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('Acne', lang='en'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q79928'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Arthritis', lang='en'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q170990'), rdflib.term.Literal('5', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Cervical spondylosis', lang='en'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q11664912'), rdflib.term.Literal('5', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Psoriasis', lang='en'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q179945'), rdflib.term.Literal('7', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Varicose veins', lang='en'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q201180'), rdflib.term.Literal('7', datatype=rdflib.term.URIRef

In [11]:
### Count Primary Symptoms per Disease from input List ###

# Demo Input
symps_list = [
    "wd:Q29971784",
    "wd:Q102187111",
    "wd:Q2971524",
    "sym:joint_pain",
    "sym:red_inflamed_skin_patches",
]

symps = " ".join(symps_list)  # -> 'wd:Q29971784 wd:Q102187111 ...'

q = f"""
PREFIX ex:   <http://example.org/med#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX sym:  <http://example.org/med#symptom/>

SELECT ?disease ?label (COUNT(DISTINCT ?in) AS ?ps_count)
WHERE {{
  VALUES ?in {{ {symps} }}
  ?disease a ex:Disease ;
           ex:hasPrimarySymptom ?in ;
           skos:prefLabel ?label .
  FILTER(lang(?label) = "en")
}}
GROUP BY ?disease ?label
ORDER BY DESC(?ps_count)
LIMIT 5
"""
for row in g.query(q):
  print(row)

(rdflib.term.URIRef('http://www.wikidata.org/entity/Q179945'), rdflib.term.Literal('Psoriasis', lang='en'), rdflib.term.Literal('5', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q464067'), rdflib.term.Literal('Fungal infection', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q30953'), rdflib.term.Literal('Dengue', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q170990'), rdflib.term.Literal('Arthritis', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))


# Test Queries

## Query 1: Exact match only (patient symptoms set matches disease symptoms set exactly)

In [12]:
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX sym: <http://example.org/med#symptom/>
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label ?matched ?total
WHERE {

  # Subquery 1: #_matched_symps
  {
    SELECT ?disease (COUNT(DISTINCT ?in) AS ?matched)
    WHERE{
      VALUES ?in {
        wd:Q352585        # frequent urination
        wd:Q1124286       # polyuria
        wd:Q635195        # polydipsia
        wd:Q653267        # polyphagia
        wd:Q4930803       # blurred vision
        wd:Q2078852       # chronic wound
        wd:Q9690          # fatigue
        wd:Q35805         # cough
        wd:Q81938         # pain
        wd:Q165947        # hunger
        wd:Q1292082       # sore throat
      }

      ?disease a ex:Disease ;
            ex:hasPrimarySymptom|ex:hasSecondarySymptom ?in .
    }

    GROUP BY ?disease
  }

  # Subquery 2: #_total_symps
  {
    SELECT ?disease (COUNT(DISTINCT ?symptom) AS ?total)
    WHERE{

      ?disease a ex:Disease ;
           ex:hasPrimarySymptom|ex:hasSecondarySymptom ?symptom .
    }
    GROUP BY ?disease
  }

  # Subquery 3: #_symps_in
  {
    SELECT( COUNT(*) AS ?inputs)
    WHERE{
      VALUES ?in {
        wd:Q352585        # frequent urination
        wd:Q1124286       # polyuria
        wd:Q635195        # polydipsia
        wd:Q653267        # polyphagia
        wd:Q4930803       # blurred vision
        wd:Q2078852       # chronic wound
        wd:Q9690          # fatigue
        wd:Q35805         # cough
        wd:Q81938         # pain
        wd:Q165947        # hunger
        wd:Q1292082       # sore throat
      }
    }
}
FILTER(?total = ?matched && ?matched = ?inputs)

?disease skos:prefLabel ?label .
}

LIMIT 10
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('Diabetes mellitus', lang='en'), rdflib.term.Literal('11', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('11', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))


## Query 2: Patient symptoms must be a subset of disease symptoms (no extra symptoms)

In [13]:
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX sym: <http://example.org/med#symptom/>
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label ?matched
WHERE {

  # Subquery 1: #_matched_symps
  {
    SELECT ?disease (COUNT(DISTINCT ?in) AS ?matched)
    WHERE{
      VALUES ?in {
        wd:Q9690
        wd:Q81938
        #wd:Q29971784
      }

      ?disease a ex:Disease ;
            ex:hasPrimarySymptom|ex:hasSecondarySymptom ?in .
    }

    GROUP BY ?disease
  }

  # Subquery 3: #_symps_in
  {
    SELECT( COUNT(*) AS ?inputs)
    WHERE{
      VALUES ?in {
        wd:Q9690
        wd:Q81938
        #wd:Q29971784
      }
    }
}
FILTER(?matched = ?inputs)

?disease skos:prefLabel ?label .
}


LIMIT 10
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('Psoriasis', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Varicose veins', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Typhoid', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Chickenpox', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Impetigo', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Dengue', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Common cold', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSc

## Query 3: Disease must have at least N symptoms from the input symptom list

In [15]:
q = """
PREFIX ex: <http://example.org/med#>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX sym: <http://example.org/med#symptom/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label (COUNT(DISTINCT ?symptoms) AS ?matched_symptoms)
WHERE {
  VALUES ?symptoms {
      wd:Q86     # headache
      wd:Q81938  # pain
  }
  ?disease a ex:Disease .
  ?disease skos:prefLabel ?label .
  ?disease ex:hasPrimarySymptom|ex:hasSecondarySymptom ?symptoms .

}
GROUP BY ?label
HAVING (COUNT(DISTINCT ?symptoms) >= 2)
ORDER BY ASC(?matched_symptoms)
LIMIT 10
"""
for row in g.query(q):
    print(row)


(rdflib.term.Literal('Typhoid', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Chickenpox', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Impetigo', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Dengue', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Fungal infection', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Common cold', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Hypertension', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/

## Query 4: Find rare symptoms (input symptoms present in less diseases are more informative)

In [33]:
q = """
PREFIX ex:   <http://example.org/med#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX sym:  <http://example.org/med#symptom/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?slabel ?sf
WHERE {
  # user-provided symptom IRIs
  VALUES ?s {
    sym:red_inflamed_skin_patches
    wd:Q9690
    wd:Q81938
    wd:Q29971784
    wd:Q102187111
    sym:joint_pain
    wd:Q2971524
  }

  ?s skos:prefLabel ?slabel .

  # sf = number of diseases that have symptom ?s
  {
    SELECT ?s (COUNT(DISTINCT ?d) AS ?sf)
    WHERE {
      ?d rdf:type ex:Disease .
      ?d ex:hasPrimarySymptom|ex:hasSecondarySymptom ?s .
    }
    GROUP BY ?s
  }
}
ORDER BY DESC(?sf)
"""
for row in g.query(q):
    print(row)

(rdflib.term.Literal('pain', lang='en'), rdflib.term.Literal('22', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('fatigue', lang='en'), rdflib.term.Literal('14', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('joint pain', lang='en'), rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('nail discoloration', lang='en'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('red, inflamed skin patches', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('nail pitting', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('eye inflammation', lang='en'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef

## Query 5: Coverage between input set and deseases' symptom sets (matches/num_input_symps)

In [36]:
q = """
PREFIX ex:  <http://example.org/med#>
PREFIX sym: <http://example.org/med#symptom/>
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label ?score ?matched ?total
WHERE {

  # Subquery 1: #_matched_symptoms
  {
    SELECT ?disease (COUNT(DISTINCT ?in) AS ?matched)
    WHERE{
      VALUES ?in {
        sym:red_inflamed_skin_patches
        wd:Q9690
        wd:Q81938
      }

      ?disease a ex:Disease ;
            ex:hasPrimarySymptom|ex:hasSecondarySymptom ?in .
    }

    GROUP BY ?disease
  }

  # Subquery 2: #_total_symptoms
  {
    SELECT ?disease (COUNT(DISTINCT ?symptom) AS ?total)
    WHERE{

      ?disease a ex:Disease ;
           ex:hasPrimarySymptom|ex:hasSecondarySymptom ?symptom .
    }
    GROUP BY ?disease
  }

  BIND( xsd:decimal(?matched) / xsd:decimal(?total) AS ?score )

  ?disease skos:prefLabel ?label .

}

ORDER BY DESC(?score)

LIMIT 5
"""

for row in g.query(q):
  print(row)

(rdflib.term.Literal('Psoriasis', lang='en'), rdflib.term.Literal('0.4285714285714285714285714286', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('7', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Varicose veins', lang='en'), rdflib.term.Literal('0.2857142857142857142857142857', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('7', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Impetigo', lang='en'), rdflib.term.Literal('0.25', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdfli

## Query 6: Top k Diseases by Score (Included in Final Deliverable with small changes) ##

In [37]:
symps_list = [
   "wd:Q9690",           # fatigue
   "wd:Q38933",          # fever
   "wd:Q81938",          # pain
   "wd:Q186889",         # nausea
   "wd:Q127076"          # vomiting
]

exclude_label = "Typhoid" ### EXCLUDE THIS DISEASE GIVEN BY LABEL (not part of the final deliverable)

top_k_results = 3

input_symps = " ".join(symps_list)

q = f"""
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX ex:   <http://example.org/med#>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX sym:  <http://example.org/med#symptom/>

SELECT ?disease ?label
       ?catNorm
       ?baseScore ?finalScore
WHERE {{

  ### USER INPUT SYMPTOMS ###
  VALUES ?inSym {{ {input_symps} }}

  ### DISEASE + LABEL ###
  ?disease a ex:Disease ;
           skos:prefLabel ?label .
  FILTER(lang(?label) = "en")
  FILTER(LCASE(STR(?label)) != LCASE("Typhoid"))



  ### COUNT INPUT SYMPTOMS ###
  {{
    SELECT (COUNT(DISTINCT ?in) AS ?inputCount)
    WHERE {{
      VALUES ?in {{ {input_symps} }}
    }}
  }}

  ### MATCHED COUNT: COUNT INPUT SYMPTOMS EXISTING IN DEASEASE SYMPTOMATOLOGY ###
  {{
    SELECT ?disease (COUNT(DISTINCT ?m) AS ?matchedCount)
    WHERE {{
      VALUES ?m {{ {input_symps} }}
      ?disease (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?m .
    }}
    GROUP BY ?disease
  }}

  ### DISEASE TOTAL SYMPTOMS ###
  {{
    SELECT ?disease (COUNT(DISTINCT ?sAll) AS ?diseaseSymptomCount)
    WHERE {{
      ?disease (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?sAll .
    }}
    GROUP BY ?disease
  }}

  ### JACCARD SIMILARITY ###
  BIND(
    IF(
      (xsd:decimal(?inputCount) + xsd:decimal(?diseaseSymptomCount) - xsd:decimal(?matchedCount)) = 0,
      0.0,
      xsd:decimal(?matchedCount) /
      (xsd:decimal(?inputCount) + xsd:decimal(?diseaseSymptomCount) - xsd:decimal(?matchedCount))
    ) AS ?jaccard
  )

  ### COVERAGE ###
  BIND(
    IF(
      xsd:decimal(?diseaseSymptomCount) = 0,
      0.0,
      xsd:decimal(?matchedCount) / xsd:decimal(?diseaseSymptomCount)
    ) AS ?coverage
  )


  ### RARE SYMPTOM CONTRIBUTION PER DISEASE ###

  ### IDF RAW: idfRaw(disease) = 1/df(m), df(m) = #diseases containing symptom m ###

  {{
    SELECT ?disease (SUM(?w) AS ?idfRaw)
    WHERE {{
      VALUES ?t {{ {input_symps} }}
      ?disease (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?t .

      {{
        SELECT ?t (COUNT(DISTINCT ?d2) AS ?df)
        WHERE {{
          ?d2 a ex:Disease ;
              (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?t .
        }}
        GROUP BY ?t
      }}

      BIND( IF(?df = 0, 0.0, (1.0 / xsd:decimal(?df))) AS ?w )
    }}
    GROUP BY ?disease
  }}

  ### IDF MAX: idfMax = SUM_ ( x in input) --- (1/df(x))  (upper bound for this input) ###
  {{
    SELECT (SUM(?wIn) AS ?idfMax)
    WHERE {{
      VALUES ?x {{ {input_symps} }}

      {{
        SELECT ?x (COUNT(DISTINCT ?d3) AS ?dfIn)
        WHERE {{
          ?d3 a ex:Disease ;
              (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?x .
        }}
        GROUP BY ?x
      }}

      BIND( IF(?dfIn = 0, 0.0, (1.0 / xsd:decimal(?dfIn))) AS ?wIn )
    }}
  }}

  BIND( IF(?idfMax = 0, 0.0, xsd:decimal(?idfRaw) / xsd:decimal(?idfMax)) AS ?idfNorm )


  ### PRIMARY SYMPTOMS MENTIONED IN USER INPUT ###
  OPTIONAL {{
    SELECT ?disease (COUNT(DISTINCT ?cm) AS ?coreMatchedRaw)
    WHERE {{
      VALUES ?cm {{ {input_symps} }}
      ?disease ex:hasPrimarySymptom ?cm .
    }}
    GROUP BY ?disease
  }}
  BIND(COALESCE(?coreMatchedRaw, 0) AS ?coreMatched)

  BIND(
    IF(?coreMatched >= 3, 1.00,
      IF(?coreMatched = 2, 0.90,
        IF(?coreMatched = 1, 0.70, 0.40)
      )
    ) AS ?coreCoeff
  )

  ### SYMPTOM TYPE BONUS ###
  {{
    SELECT (COUNT(DISTINCT ?pType) AS ?patientCatCount)
    WHERE {{
      VALUES ?s {{ {input_symps} }}
      ?s a ?pType .
      ?pType rdfs:subClassOf ex:Symptom .
    }}
  }}

  OPTIONAL {{
    SELECT ?disease (COUNT(DISTINCT ?overCat) AS ?catOverlapRaw)
    WHERE {{
      VALUES ?s {{ {input_symps} }}

      ?s a ?pType .
      ?pType rdfs:subClassOf ex:Symptom .

      ?disease (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?ds .
      ?ds a ?dType .
      ?dType rdfs:subClassOf ex:Symptom .

      FILTER(?pType = ?dType)
      BIND(?pType AS ?overCat)
    }}
    GROUP BY ?disease
  }}
  BIND(COALESCE(?catOverlapRaw, 0) AS ?catOverlap)

  BIND(
    IF(?patientCatCount = 0, 0.0,
      xsd:decimal(?catOverlap) / xsd:decimal(?patientCatCount)
    ) AS ?catNorm
  )

  ### TOTAL SCORE FORMULA ###
  BIND( (0.6 * ?idfNorm + 0.2 * ?jaccard + 0.2 * ?coverage) AS ?baseScore )
  BIND( (?baseScore * ?coreCoeff + 0.10 * ?catNorm) AS ?finalScore )
}}
GROUP BY ?disease

ORDER BY DESC(?finalScore)
LIMIT {top_k_results}
"""

results=[]
for row in g.query(q):
    print(f"Disease:{row[1]}   Score:{float(row[-1]):.3f}")
    results.append(f"<{row[0]}>")


Disease:jaundice   Score:0.429
Disease:Malaria   Score:0.393
Disease:Urinary tract infection   Score:0.362


## Query 7: Give wiki info on Top 1 Disease (Included in Final Deliverable) ##

In [38]:
disease = "<http://www.wikidata.org/entity/Q83319>" #using previous example Typhoid's URI

q = f"""
PREFIX ex:  <http://example.org/med#>
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?label ?explanation ?visitDoctor ?treatment
WHERE {{
      {disease} a ex:Disease ;
                skos:prefLabel ?label ;
                ex:hasExplanation ?explanation ;
                ex:seeDoctor ?visitDoctor ;
                ex:treatment ?treatment .
}}
"""

for row in g.query(q):
  print(f"Disease: {row[0]}\nExplanation: {row[1]}\nVisit to the doctor: {row[2]}\nTreatment: {row[3]}")

Disease: Typhoid
Explanation: Typhoid is a serious bacterial infection spread through contaminated food or water.
Visit to the doctor: Immediately if typhoid is suspected.
Treatment: Immediate medical treatment is required; do not attempt self-treatment.


## Query 8: Return matching and missing symptoms from patients description for specific disease ##

In [39]:
# Matching Symptoms #
disease = "<http://www.wikidata.org/entity/Q83319>" #using previous example Typhoid's URI


symps_list = [
     "wd:Q2536390" ,       # abdominal distention
     "wd:Q29644032" ,      # continuous fever
     "wd:Q38933",          # fever
     "wd:Q9690" ,          # fatigue
     "wd:Q86" ,            # headache
     "wd:Q87"              # NOT PART OF TYPHOIDS SYMPTOMS
]

input_symps = " ".join(symps_list)



q = f"""
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX ex:   <http://example.org/med#>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX sym:  <http://example.org/med#symptom/>

SELECT ?label ?SympName

WHERE {{

  ### USER INPUT SYMPTOMS ###
  VALUES ?inSym {{ {input_symps} }}

  ### DISEASE + LABEL ###
  {disease} a ex:Disease ;
           skos:prefLabel ?label .
  FILTER(lang(?label) = "en")

  ### FIND SYMPTOMS MENTIONED BY THE USER ###
  {disease} (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?inSym .
  ?inSym skos:prefLabel ?SympName .

}}
GROUP BY ?SympName
ORDER BY ?SympName
LIMIT 25
"""

for row in g.query(q):
    print(f"{row[0]}'s Matched Symptom: {row[1]}")

Typhoid's Matched Symptom: abdominal distention
Typhoid's Matched Symptom: continuous fever
Typhoid's Matched Symptom: fatigue
Typhoid's Matched Symptom: fever
Typhoid's Matched Symptom: headache


In [40]:
# Missing Symptoms #
disease = "<http://www.wikidata.org/entity/Q83319>" #using previous example Typhoid's URI


symps_list = [
     "wd:Q2536390" ,       # abdominal distention
     "wd:Q29644032" ,      # continuous fever
     "wd:Q38933",          # fever
     "wd:Q9690" ,          # fatigue
     "wd:Q86" ,            # headache
     "wd:Q87"              # NOT PART OF TYPHOIDS SYMPTOMS
]

input_symps = " ".join(symps_list)



q = f"""
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX ex:   <http://example.org/med#>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX sym:  <http://example.org/med#symptom/>

SELECT ?label ?SympName
WHERE {{

  ### USER INPUT SYMPTOMS ###
  VALUES ?inSym {{ {input_symps} }}

  ### DISEASE + LABEL ###
  {disease} a ex:Disease ;
           skos:prefLabel ?label .
  FILTER(lang(?label) = "en")

  ### FIND SYMPTOMS MENTIONED BY THE USER ###
  {disease} (ex:hasPrimarySymptom|ex:hasSecondarySymptom) ?symptom .
  ?symptom skos:prefLabel ?SympName .

  # NEGATION: Keep only symptoms NOT in the user's list
  MINUS {{
    VALUES ?symptom {{ {input_symps} }}
  }}
}}
GROUP BY ?SympName
ORDER BY ?SympName
LIMIT 25
"""

for row in g.query(q):
    # row[0] is the disease label, row[1] is the missing symptom name
    print(f"{row[0]}'s Missing Symptom: {row[1]}")

Typhoid's Missing Symptom: bradycardia
Typhoid's Missing Symptom: confusion or consciousness
Typhoid's Missing Symptom: constipation
Typhoid's Missing Symptom: delusion
Typhoid's Missing Symptom: diarrhea
Typhoid's Missing Symptom: gastrointestinal bleeding
Typhoid's Missing Symptom: hepatomegaly
Typhoid's Missing Symptom: insomnia
Typhoid's Missing Symptom: intestinal perforation
Typhoid's Missing Symptom: leukopenia
Typhoid's Missing Symptom: loss of appitite
Typhoid's Missing Symptom: nausea
Typhoid's Missing Symptom: pain
Typhoid's Missing Symptom: pallor
Typhoid's Missing Symptom: prostration
Typhoid's Missing Symptom: rash
Typhoid's Missing Symptom: splenomegaly
Typhoid's Missing Symptom: vomiting
